# Phân hệ 1–2 & Thí nghiệm 1: Huấn luyện ConvNeXt-Tiny + MC Dropout
Notebook tự động tải PlantVillage (10 lớp cà chua) + PlantDoc, huấn luyện mô hình có Dropout, chạy Thí nghiệm 1 (Softmax vs Temperature Scaling vs MC Dropout) và xuất `artifacts/`.

In [ ]:
import json, os, random, shutil, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image, ImageEnhance, ImageFilter
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

SMOKE = False  # Đặt True để chạy kiểm tra nhanh (20 ảnh/lớp, 1 epoch)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| SMOKE:', SMOKE)

CLASSES = [
    'Bacterial_spot', 'Early_blight', 'Late_blight', 'Leaf_Mold', 'Septoria_leaf_spot',
    'Spider_mites', 'Target_Spot', 'Yellow_Leaf_Curl_Virus', 'Mosaic_virus', 'healthy'
]
ARTIFACTS = Path('artifacts')
ARTIFACTS.mkdir(exist_ok=True)

In [ ]:
# Cài đặt các hàm từ plant_expert.vision (tự động dùng file repo nếu có, hoặc định nghĩa trực tiếp trên Colab)
try:
    from plant_expert.vision import build_model, enable_mc_dropout, mc_predict, predictive_entropy, mutual_info
except ImportError:
    EPS = 1e-12
    def build_model(num_classes: int, p_drop: float = 0.3, pretrained: bool = False) -> nn.Module:
        weights = models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
        m = models.convnext_tiny(weights=weights)
        in_features = m.classifier[2].in_features
        m.classifier[2] = nn.Sequential(nn.Dropout(p_drop), nn.Linear(in_features, num_classes))
        return m
    def enable_mc_dropout(model: nn.Module) -> None:
        model.eval()
        for mod in model.modules():
            if isinstance(mod, nn.Dropout):
                mod.train()
    @torch.no_grad()
    def mc_predict(model: nn.Module, x: torch.Tensor, T: int = 20) -> np.ndarray:
        enable_mc_dropout(model)
        return np.stack([torch.softmax(model(x), dim=-1).cpu().numpy() for _ in range(T)], axis=0)
    def predictive_entropy(p_mean: np.ndarray) -> np.ndarray:
        p = np.asarray(p_mean, dtype=float)
        return -(p * np.log(p + EPS)).sum(axis=-1)
    def mutual_info(probs: np.ndarray) -> np.ndarray:
        probs = np.asarray(probs, dtype=float)
        h_mean = predictive_entropy(probs.mean(axis=0))
        mean_h = -(probs * np.log(probs + EPS)).sum(axis=-1).mean(axis=0)
        return np.maximum(h_mean - mean_h, 0.0)

In [ ]:
# Tải dữ liệu PlantVillage (10 lớp cà chua) và PlantDoc
DATA_DIR = Path('data')
PV_DIR = DATA_DIR / 'PlantVillage-Dataset'
PD_DIR = DATA_DIR / 'PlantDoc-Dataset'
DATA_DIR.mkdir(exist_ok=True)

if not PV_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse',
                    'https://github.com/spMohanty/PlantVillage-Dataset.git', str(PV_DIR)], check=True)
    subprocess.run(['git', '-C', str(PV_DIR), 'sparse-checkout', 'set', 'raw/color'], check=True)

if not PD_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/pratikkayal/PlantDoc-Dataset.git', str(PD_DIR)], check=True)

PV_MAP = {
    'Tomato___Bacterial_spot': 'Bacterial_spot',
    'Tomato___Early_blight': 'Early_blight',
    'Tomato___Late_blight': 'Late_blight',
    'Tomato___Leaf_Mold': 'Leaf_Mold',
    'Tomato___Septoria_leaf_spot': 'Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite': 'Spider_mites',
    'Tomato___Target_Spot': 'Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 'Yellow_Leaf_Curl_Virus',
    'Tomato___Tomato_mosaic_virus': 'Mosaic_virus',
    'Tomato___healthy': 'healthy',
}
PD_MAP = {
    'Tomato leaf bacterial spot': 'Bacterial_spot',
    'Tomato Early blight leaf': 'Early_blight',
    'Tomato leaf late blight': 'Late_blight',
    'Tomato mold leaf': 'Leaf_Mold',
    'Tomato Septoria leaf spot': 'Septoria_leaf_spot',
    'Tomato two spotted spider mites leaf': 'Spider_mites',
    'Tomato leaf yellow virus': 'Yellow_Leaf_Curl_Virus',
    'Tomato leaf mosaic virus': 'Mosaic_virus',
    'Tomato leaf': 'healthy',
}

pv_items = []
for folder, cls in PV_MAP.items():
    p = PV_DIR / 'raw' / 'color' / folder
    imgs = sorted([x for x in p.glob('*') if x.suffix.lower() in ('.jpg', '.jpeg', '.png')])
    if SMOKE:
        imgs = imgs[:20]
    pv_items.extend([(str(x), cls) for x in imgs])

pd_items = []
for split_sub in ['train', 'test']:
    for folder, cls in PD_MAP.items():
        p = PD_DIR / split_sub / folder
        if p.exists():
            imgs = sorted([x for x in p.glob('*') if x.suffix.lower() in ('.jpg', '.jpeg', '.png')])
            if SMOKE:
                imgs = imgs[:5]
            pd_items.extend([(str(x), cls) for x in imgs])

labels = [c for _, c in pv_items]
train_items, tmp_items = train_test_split(pv_items, test_size=0.30, stratify=labels, random_state=SEED)
val_items, test_items = train_test_split(tmp_items, test_size=0.50, stratify=[c for _, c in tmp_items], random_state=SEED)
print(f'Train: {len(train_items)}, Val: {len(val_items)}, Test: {len(test_items)}, PlantDoc: {len(pd_items)}')

In [ ]:
# Dataset, các phép biến đổi nhiễu (blur, occlusion, brightness) và huấn luyện
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
CLS_IDX = {c: i for i, c in enumerate(CLASSES)}

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

def apply_corruption(img: Image.Image, mode: str) -> Image.Image:
    if mode == 'blur':
        return img.filter(ImageFilter.GaussianBlur(radius=3.5))
    if mode == 'brightness':
        return ImageEnhance.Brightness(img).enhance(0.45)
    if mode == 'occlusion':
        out = img.copy()
        w, h = out.size
        patch = Image.new('RGB', (int(w * 0.4), int(h * 0.4)), (30, 30, 30))
        out.paste(patch, (int(w * 0.3), int(h * 0.3)))
        return out
    return img

class LeafDataset(Dataset):
    def __init__(self, items, tf, corruption='clean'):
        self.items = items; self.tf = tf; self.corruption = corruption
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, cls = self.items[i]
        img = Image.open(path).convert('RGB')
        img = apply_corruption(img, self.corruption)
        return self.tf(img), CLS_IDX[cls], f'{self.corruption}:{Path(path).stem}_{i}'

BATCH = 32
train_loader = DataLoader(LeafDataset(train_items, train_tf), batch_size=BATCH, shuffle=True)
val_loader = DataLoader(LeafDataset(val_items, eval_tf), batch_size=BATCH)

model = build_model(len(CLASSES), p_drop=0.3, pretrained=not SMOKE).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
crit = nn.CrossEntropyLoss()
EPOCHS = 1 if SMOKE else 8
best_acc = -1.0

for ep in range(EPOCHS):
    model.train()
    for x, y, _ in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step()
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y, _ in val_loader:
            pred = model(x.to(DEVICE)).argmax(dim=-1).cpu()
            correct += (pred == y).sum().item(); total += len(y)
    acc = correct / max(total, 1)
    print(f'Epoch {ep+1}/{EPOCHS} - val_acc={acc:.4f}')
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), ARTIFACTS / 'model.pt')

model.load_state_dict(torch.load(ARTIFACTS / 'model.pt', map_location=DEVICE, weights_only=True))

In [ ]:
# Temperature Scaling trên tập Validation + các hàm tính ECE, AUROC, AURC
@torch.no_grad()
def collect_logits(loader):
    model.eval(); logits, ys, ids = [], [], []
    for x, y, item_ids in loader:
        logits.append(model(x.to(DEVICE)).cpu()); ys.append(y); ids.extend(item_ids)
    return torch.cat(logits), torch.cat(ys).numpy(), ids

val_logits, val_y, _ = collect_logits(val_loader)
temp = nn.Parameter(torch.ones(1) * 1.5)
lbfgs = torch.optim.LBFGS([temp], lr=0.05, max_iter=50)
def _closure():
    lbfgs.zero_grad()
    loss = nn.CrossEntropyLoss()(val_logits / temp.clamp(min=0.05), torch.tensor(val_y))
    loss.backward()
    return loss
lbfgs.step(_closure)
T_OPT = float(temp.clamp(min=0.05).item())
print('Optimal Temperature T_opt =', round(T_OPT, 4))

def compute_ece(probs: np.ndarray, y: np.ndarray, n_bins: int = 15) -> float:
    conf = probs.max(axis=1)
    acc = (probs.argmax(axis=1) == y).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(ece)

def risk_coverage(u: np.ndarray, err: np.ndarray):
    order = np.argsort(u)
    sorted_err = err[order]
    cov = np.arange(1, len(u) + 1) / len(u)
    risk = np.cumsum(sorted_err) / np.arange(1, len(u) + 1)
    return cov, risk, float(np.trapezoid(risk, cov))

In [ ]:
# Chạy 3 phương pháp (softmax, temp_scaling, mc_dropout) trên Val + các tập Test + chọn ngưỡng tau
T_MC = 20

def eval_loader_all(loader):
    logits, y, ids = collect_logits(loader)
    p_soft = torch.softmax(logits, dim=-1).numpy()
    p_temp = torch.softmax(logits / T_OPT, dim=-1).numpy()
    mc_list = []
    for x, _, _ in loader:
        mc_list.append(mc_predict(model, x.to(DEVICE), T=T_MC))
    probs_mc = np.concatenate(mc_list, axis=1)  # [T, N, C]
    p_mc = probs_mc.mean(axis=0)
    return y, ids, {
        'softmax': (p_soft, predictive_entropy(p_soft), None),
        'temp_scaling': (p_temp, predictive_entropy(p_temp), None),
        'mc_dropout': (p_mc, predictive_entropy(p_mc), mutual_info(probs_mc)),
    }

val_y, _, val_res = eval_loader_all(val_loader)
val_p_mc, val_H_mc, _ = val_res['mc_dropout']
val_err = (val_p_mc.argmax(axis=1) != val_y).astype(float)
cov, risk, _ = risk_coverage(val_H_mc, val_err)
sorted_H = np.sort(val_H_mc)
ok = np.where(risk <= 0.05)[0]
TAU = float(sorted_H[ok[-1]]) if len(ok) else float(np.quantile(val_H_mc, 0.80))
print('Selected threshold tau =', round(TAU, 4))

cfg = {
    'classes': CLASSES,
    'tau': TAU,
    'p_drop': 0.3,
    'img_size': IMG_SIZE,
    'mean': MEAN,
    'std': STD,
    'T': T_MC,
}
(ARTIFACTS / 'config.json').write_text(json.dumps(cfg, indent=2), encoding='utf-8')

In [ ]:
# Thí nghiệm 1 trên các tập clean, blur, occlusion, brightness, plantdoc + xuất CSV & 3 biểu đồ
splits = {
    'clean': DataLoader(LeafDataset(test_items, eval_tf, 'clean'), batch_size=BATCH),
    'blur': DataLoader(LeafDataset(test_items, eval_tf, 'blur'), batch_size=BATCH),
    'occlusion': DataLoader(LeafDataset(test_items, eval_tf, 'occlusion'), batch_size=BATCH),
    'brightness': DataLoader(LeafDataset(test_items, eval_tf, 'brightness'), batch_size=BATCH),
}
if len(pd_items) > 0:
    splits['plantdoc'] = DataLoader(LeafDataset(pd_items, eval_tf, 'plantdoc'), batch_size=BATCH)

metric_rows = []
mc_rows = []
plot_store = {}

for sname, loader in splits.items():
    y, ids, res = eval_loader_all(loader)
    plot_store[sname] = (y, res)
    for mname, (p, H, mi) in res.items():
        err = (p.argmax(axis=1) != y).astype(int)
        auroc = roc_auc_score(err, H) if len(np.unique(err)) > 1 else float('nan')
        auroc_mi = roc_auc_score(err, mi) if (mi is not None and len(np.unique(err)) > 1) else float('nan')
        _, _, aurc = risk_coverage(H, err.astype(float))
        metric_rows.append({
            'split': sname, 'method': mname,
            'acc': float((1 - err).mean()),
            'ece': compute_ece(p, y),
            'auroc_err': float(auroc),
            'auroc_mi': float(auroc_mi),
            'aurc': aurc,
        })
    p_mc, H_mc, mi_mc = res['mc_dropout']
    for i, img_id in enumerate(ids):
        row = {'image_id': img_id, 'split': sname, 'label': CLASSES[y[i]], 'H': float(H_mc[i]), 'MI': float(mi_mc[i])}
        for c_i, c_name in enumerate(CLASSES):
            row[f'p_{c_name}'] = float(p_mc[i, c_i])
        mc_rows.append(row)

pd.DataFrame(metric_rows).to_csv(ARTIFACTS / 'exp1_metrics.csv', index=False)
pd.DataFrame(mc_rows).to_csv(ARTIFACTS / 'test_mc_probs.csv', index=False)

# Hình 1: Reliability diagram trên tập blur
y_b, res_b = plot_store['blur']
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect')
for mname, (p, _, _) in res_b.items():
    conf, acc = p.max(1), (p.argmax(1) == y_b).astype(float)
    bins = np.linspace(0, 1, 11)
    xs, ys_b = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            xs.append(conf[m].mean()); ys_b.append(acc[m].mean())
    ax.plot(xs, ys_b, marker='o', label=mname)
ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy'); ax.set_title('Reliability (blur)'); ax.legend()
fig.tight_layout(); fig.savefig(ARTIFACTS / 'exp1_reliability.png', dpi=150); plt.close(fig)

# Hình 2: Risk-Coverage trên tập blur
fig, ax = plt.subplots(figsize=(5, 4))
for mname, (p, H, _) in res_b.items():
    err = (p.argmax(1) != y_b).astype(float)
    c_curve, r_curve, _ = risk_coverage(H, err)
    ax.plot(c_curve, r_curve, label=mname)
ax.set_xlabel('Coverage'); ax.set_ylabel('Risk (Error rate)'); ax.set_title('Risk-Coverage (blur)'); ax.legend()
fig.tight_layout(); fig.savefig(ARTIFACTS / 'exp1_risk_coverage.png', dpi=150); plt.close(fig)

# Hình 3: Phân phối Entropy đúng vs sai của MC Dropout
p_mc, H_mc, _ = res_b['mc_dropout']
ok_m = p_mc.argmax(1) == y_b
fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(H_mc[ok_m], bins=20, alpha=0.6, label='Correct')
ax.hist(H_mc[~ok_m], bins=20, alpha=0.6, label='Wrong')
ax.axvline(TAU, color='red', linestyle='--', label=f'tau={TAU:.2f}')
ax.set_xlabel('Predictive Entropy H'); ax.set_ylabel('Count'); ax.legend()
fig.tight_layout(); fig.savefig(ARTIFACTS / 'exp1_entropy_hist.png', dpi=150); plt.close(fig)

shutil.make_archive('artifacts', 'zip', str(ARTIFACTS))
print('Saved all artifacts and artifacts.zip')